In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(os.path.abspath("."))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATASET_DIR = (PROJECT_ROOT / "data" / "research_processed_smoke_auto").resolve()
MODEL_DIR = (PROJECT_ROOT / "models" / "research_smoke_auto").resolve()
RESULTS_DIR = (PROJECT_ROOT / "results" / "research_smoke_auto").resolve()

# 04 GPT

GPT adjudication only runs on confirmed anomalies from the selected live/evaluation mode.
The compact payload includes the mode, component scores, threshold context, and dominant features.

In [2]:
import pandas as pd

from src.config import GPTConfig
from src.gpt_adjudicator import adjudicate_anomaly_records, build_window_summary, call_openai_responses_api
from src.utils import read_json

In [3]:
cfg = GPTConfig(
    evaluation_dir=RESULTS_DIR,
    output_dir=RESULTS_DIR,
    max_records=20,
)

confirmed_alerts = pd.read_csv(RESULTS_DIR / "realtime_alert_candidates.csv")
confirmed_alerts.head()

,window_id,container_id,machine_id,end_time,split,anomaly_score,threshold,score_over_threshold,feature_error_vector,top_feature_rank,...,z_score_reason,current_consecutive_breach_count,score_buffer_size,decision,status,anomaly_candidate,confirmed_anomaly,decision_reason,compact_anomaly_summary,gpt_triggered
0,149,unknown_container,unknown_machine,149,test,0.407054,0.874815,0.263941,"[0.04266877844929695, 0.008643602021038532, 0....","[4, 7, 2, 3, 5, 0, 6, 1]",...,NaN,3,150,confirmed_anomaly,confirmed_anomaly,True,True,"smoothed_score_above_threshold,consecutive_bre...","{""window_id"": 149, ""container_id"": ""unknown_co...",False
1,331,unknown_container,unknown_machine,331,test,0.501030,2.311735,1.729172,"[0.022084934636950493, 0.0016546942060813308, ...","[4, 7, 2, 3, 0, 6, 1, 5]",...,NaN,3,200,confirmed_anomaly,confirmed_anomaly,True,True,"smoothed_score_above_threshold,consecutive_bre...","{""window_id"": 331, ""container_id"": ""unknown_co...",False
2,504,unknown_container,unknown_machine,504,test,0.268445,3.848743,5.356625,"[0.02518528699874878, 0.01152642723172903, 0.0...","[4, 7, 2, 0, 3, 1, 5, 6]",...,NaN,3,200,confirmed_anomaly,confirmed_anomaly,True,True,"smoothed_score_above_threshold,consecutive_bre...","{""window_id"": 504, ""container_id"": ""unknown_co...",False
3,755,unknown_container,unknown_machine,755,test,0.355630,3.828679,0.266191,"[0.2145152986049652, 0.20839054882526398, 0.18...","[4, 3, 7, 5, 0, 1, 2, 6]",...,NaN,3,200,confirmed_anomaly,confirmed_anomaly,True,True,"smoothed_score_above_threshold,consecutive_bre...","{""window_id"": 755, ""container_id"": ""unknown_co...",False
4,956,unknown_container,unknown_machine,956,test,0.706730,3.822841,1.429609,"[0.0484030619263649, 0.009539565071463585, 0.0...","[4, 7, 2, 0, 1, 3, 6, 5]",...,NaN,3,200,confirmed_anomaly,confirmed_anomaly,True,True,"smoothed_score_above_threshold,consecutive_bre...","{""window_id"": 956, ""container_id"": ""unknown_co...",False


In [4]:
sample_payload = confirmed_alerts.iloc[0].to_dict() if len(confirmed_alerts) > 0 else {}
sample_summary = build_window_summary(sample_payload) if sample_payload else {}
sample_summary

{'window_id': 149,
 'container_id': 'unknown_container',
 'machine_id': 'unknown_machine',
 'mode': 'hybrid',
 'split': 'test',
 'time_range': {'start_time': -1, 'end_time': 149},
 'recon_score': 0.3227143585681915,
 'forecast_score': 0.5335637927055359,
 'final_score': 0.4070541262626648,
 'anomaly_score': 0.4070541262626648,
 'threshold': 0.8748154640197754,
 'dynamic_threshold': 0.8748154640197754,
 'final_threshold': 0.8748154640197754,
 'score_over_threshold': 0.2639412879943847,
 'decision_reason': 'smoothed_score_above_threshold,consecutive_breach=3/3,confirmed_after_consecutive_breaches',
 'top_k_features': ['mpki', 'disk_io', 'cpi', 'mem_gps', 'net_in'],
 'top_k_feature_errors': [1.9633736610412598,
  0.9609150290489197,
  0.1331232190132141,
  0.07357458025217056,
  0.046759746968746185],
 'feature_error_vector': [0.04266877844929695,
  0.008643602021038532,
  0.1331232190132141,
  0.07357458025217056,
  1.9633736610412598,
  0.046759746968746185,
  0.027374381199479103,
  0.

In [5]:
if sample_summary:
    sample_decision, sample_meta = call_openai_responses_api(sample_summary, cfg)
else:
    sample_decision, sample_meta = {}, {}

sample_meta, sample_decision

({'used_fallback': True,
  'reason': 'openai_request_failed',
  'error': "Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}"},
 {'label': 'normal',
  'severity': 'low',
  'explanation': 'The hybrid score is only marginally above threshold. Top deviations: mpki, disk_io, cpi. This looks more like a false positive than an active fault.',
  'recommended_action': 'ignore'})

In [6]:
adjudication_summary = adjudicate_anomaly_records(
    prediction_csv_path=RESULTS_DIR / "window_level_predictions.csv",
    config=cfg,
    max_records=cfg.max_records,
)
adjudication_summary

{'gpt_adjudications_csv': 'C:\\Users\\kaspe\\Desktop\\Project\\project\\results\\research_smoke_auto\\gpt_adjudications.csv',
 'gpt_adjudications_json': 'C:\\Users\\kaspe\\Desktop\\Project\\project\\results\\research_smoke_auto\\gpt_adjudications.json',
 'comparison_csv': 'C:\\Users\\kaspe\\Desktop\\Project\\project\\results\\research_smoke_auto\\ae_vs_gpt_comparison.csv',
 'comparison_json': 'C:\\Users\\kaspe\\Desktop\\Project\\project\\results\\research_smoke_auto\\ae_vs_gpt_comparison.json',
 'records': [{'window_id': 18080,
   'container_id': 'unknown_container',
   'machine_id': 'unknown_machine',
   'end_time': 18080,
   'split': 'test',
   'anomaly_score': 86.73941802978516,
   'threshold': 5.061085224151611,
   'score_over_threshold': 81.67833280563354,
   'feature_error_vector': '[0.695620059967041, 0.16363315284252167, 1.8643550872802734, 628.8776245117188, 61.916324615478516, 0.12908142805099487, 0.05804814398288727, 0.21061007678508759]',
   'top_feature_rank': '[3, 4, 2, 0

In [7]:
read_json(RESULTS_DIR / "ae_vs_gpt_comparison.json")

{'records': [{'window_id': 18080,
   'container_id': 'unknown_container',
   'machine_id': 'unknown_machine',
   'mode': 'hybrid',
   'anomaly_score': 86.73941802978516,
   'ae_only_label': 'fault_candidate',
   'gpt_label': 'critical',
   'severity': 'high',
   'recommended_action': 'raise_alert',
   'explanation': 'The hybrid score is far above threshold and the largest deviations are in mem_gps, mpki, cpi. This is consistent with an active high-severity runtime issue.'},
  {'window_id': 6494,
   'container_id': 'unknown_container',
   'machine_id': 'unknown_machine',
   'mode': 'hybrid',
   'anomaly_score': 39.21830749511719,
   'ae_only_label': 'fault_candidate',
   'gpt_label': 'critical',
   'severity': 'high',
   'recommended_action': 'raise_alert',
   'explanation': 'The hybrid score is far above threshold and the largest deviations are in mem_gps, mpki, cpu_util. This is consistent with an active high-severity runtime issue.'},
  {'window_id': 502,
   'container_id': 'unknown_